In [1]:
!pip install -qU langgraph langchain-openai langchain-google-genai langchain-community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.0/256.0 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.1/146.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.7/81.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.0/256.0 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.1/146.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.7/81.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB

In [2]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [26]:
import os
import json
import subprocess
from typing import TypedDict, Annotated, List, Dict, Any
import operator

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command, interrupt
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI

# 1. LLM Մոդելի սկզբնավորում
llm = ChatOpenAI(model="gpt-4o", temperature=0.1)


# 2. State-ի սահմանում
class AgentState(TypedDict):
    dialogue: Annotated[List[Dict[str, str]], operator.add]
    current_role: str
    instruction: str
    reason: str
    question_to_ask: str
    step_count: int


# 3. Subprocess Executor Tool
def execute_python_code(code: str, filename: str = "pinn_silverbox.py") -> str:
    with open(filename, "w", encoding="utf-8") as f:
        f.write(code)

    try:
        result = subprocess.run(
            ["python3", filename],
            capture_output=True,
            text=True,
            timeout=90
        )
        if result.returncode == 0:
            return f"SUCCESS (Exit code 0):\nSTDOUT:\n{result.stdout}"
        else:
            return f"ERROR (Exit code {result.returncode}):\nSTDOUT:\n{result.stdout}\nSTDERR:\n{result.stderr}"
    except subprocess.TimeoutExpired:
        return "ERROR: Execution timed out (exceeded 90 seconds)."
    except Exception as e:
        return f"EXECUTION FAILED: {str(e)}"


# 4. Agent Nodes-ի սահմանում

def orchestrator_node(state: AgentState):
    step_count = state.get("step_count", 0) + 1

    if step_count >= 12:
        return {
            "current_role": "finish",
            "instruction": "Maximum steps reached.",
            "reason": "Step limit reached.",
            "question_to_ask": "",
            "step_count": step_count
        }

    sys_prompt = """You are the Orchestrator for Task 5 Silverbox PINN system identification.
Analyze dialogue history and choose the next role: 'scientist', 'general', 'executor', 'human', or 'finish'.

STRICT RULES FOR CHOOSING 'human':
1. AMBIGUOUS QUERY: The latest request has more than one interpretation (e.g. unstated metrics or dataset paths). You MUST choose 'human' and ask for clarification.
2. HARD DECISION: The next step will execute PyTorch PINN training code or deliver final benchmark metrics. You MUST choose 'human', state the intended action, and ask for human confirmation.

Output ONLY valid JSON:
{
  "next_role": "scientist" | "general" | "executor" | "human" | "finish",
  "instruction": "Clear instruction for scientist/general/executor",
  "question_to_ask": "Precise question for human if next_role is human, else empty string",
  "reason": "Single sentence justification"
}"""

    history_str = json.dumps(state["dialogue"], indent=2)
    response = llm.invoke([
        SystemMessage(content=sys_prompt),
        HumanMessage(content=f"Current Dialogue History:\n{history_str}")
    ])

    try:
        res_json = json.loads(response.content.strip().strip("```json").strip("```"))
        next_role = res_json.get("next_role", "finish")
        instruction = res_json.get("instruction", "")
        question_to_ask = res_json.get("question_to_ask", "")
        reason = res_json.get("reason", "")
    except Exception:
        next_role = "human"
        instruction = ""
        question_to_ask = "Should I proceed with executing the PyTorch Silverbox PINN model?"
        reason = "Parsing fallback."

    return {
        "current_role": next_role,
        "instruction": instruction,
        "question_to_ask": question_to_ask,
        "reason": reason,
        "step_count": step_count,
        "dialogue": [{"role": "Orchestrator", "content": f"Decision -> Role: {next_role} | Reason: {reason}"}]
    }


def scientist_node(state: AgentState):
    sys_prompt = """You are the Lead Scientist for the Silverbox PINN benchmark (SNLS80mV.mat).
Domain Knowledge & Physics Requirements:
- Physics: Duffing oscillator modeling displacement y and input u. State vector is (y, v) where v = y_dot.
- System Equations: y_dot = v, v_dot = alpha_theta(y, v, u), where alpha_theta is a PyTorch MLP.
- Data Excerpt: Load 'SNLS80mV.mat', extract 'V1' (input u) and 'V2' (output y). Slice [42650:46746] (4096 samples).
- Centering: Subtract mean(u) and mean(y) on this excerpt. Sampling time dt = 1.0 / 610.35.
- Initial Velocity: Compute v[0] using finite differences on y.
- Splits: Train on first 3072 samples (~5.03 s), Test on last 1024 samples (~1.68 s).
- Numerical Integration: Classical RK4 step over dt.
- Evaluation Metric: Free-run RMSE on 1024 test samples (without reading y_test during rollout) and train RMSE on 3072 samples.

Provide clear, structured mathematical guidance to the Executor."""

    instruction = state["instruction"]
    response = llm.invoke([
        SystemMessage(content=sys_prompt),
        HumanMessage(content=f"Instruction: {instruction}")
    ])
    return {"dialogue": [{"role": "Scientist", "content": response.content}]}


def general_node(state: AgentState):
    sys_prompt = "You are the General agent. Answer high-level questions."
    instruction = state["instruction"]
    response = llm.invoke([
        SystemMessage(content=sys_prompt),
        HumanMessage(content=f"Instruction: {instruction}")
    ])
    return {"dialogue": [{"role": "General Agent", "content": response.content}]}


def human_node(state: AgentState):
    """Human Node: Uses LangGraph interrupt to request human response."""
    question = state.get("question_to_ask", "Please clarify or confirm:")
    print(f"\n==================================================")
    print(f"[GRAPH INTERRUPTED - QUESTION FOR HUMAN]: {question}")
    print(f"==================================================")

    user_response = interrupt({"question": question})

    return {
        "dialogue": [{"role": "human", "content": user_response}]
    }


def executor_node(state: AgentState):
    instruction = state["instruction"]
    history_str = json.dumps(state["dialogue"], indent=2)

    sys_prompt = """You are the Executor agent. Write a complete PyTorch script for Task 5 Silverbox PINN that achieves a HIGHLY ACCURATE free-run Test RMSE (< 0.05 V).

CRITICAL PHYSICAL SCALING & INITIALIZATION RULES:
1. Device: `device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')`.
2. Data Preprocessing:
   - Load 'SNLS80mV.mat'. Extract u = V1.flatten(), y = V2.flatten().
   - Slice excerpt [42650:46746] (4096 samples). Mean-center u and y on excerpt.
   - dt = 1.0 / 610.35. Compute v = np.gradient(y, dt).
   - Train = first 3072, Test = last 1024. Convert to float32 tensors on `device`.

3. Physics-Aware Model Architecture (Input Normalization & Zero-Init):
   - Define `DuffingMLP(nn.Module)`:
     ```python
     class DuffingMLP(nn.Module):
         def __init__(self):
             super().__init__()
             self.net = nn.Sequential(
                 nn.Linear(3, 64),
                 nn.Tanh(),
                 nn.Linear(64, 64),
                 nn.Tanh(),
                 nn.Linear(64, 1)
             )
             # ZERO-INIT final layer so initial acceleration predictions are tiny!
             nn.init.zeros_(self.net[-1].weight)
             nn.init.zeros_(self.net[-1].bias)

         def forward(self, y, v, u):
             # CRITICAL: Scale inputs so y, v, u are all ~ O(1)
             y_scaled = (y.reshape(-1, 1)) / 0.2
             v_scaled = (v.reshape(-1, 1)) / 50.0
             u_scaled = (u.reshape(-1, 1)) / 0.2
             x = torch.cat([y_scaled, v_scaled, u_scaled], dim=1)
             return 200.0 * torch.tanh(self.net(x))
     ```

4. RK4 Step & Free-Run Integration:
   - Define `rk4_step(model, y, v, u, dt)`:
     `k1_v = model(y, v, u).squeeze()`
     `k1_y = v`
     `k2_v = model(y + 0.5*dt*k1_y, v + 0.5*dt*k1_v, u).squeeze()`
     `k2_y = v + 0.5*dt*k1_v`
     `k3_v = model(y + 0.5*dt*k2_y, v + 0.5*dt*k2_v, u).squeeze()`
     `k3_y = v + 0.5*dt*k2_v`
     `k4_v = model(y + dt*k3_y, v + dt*k3_v, u).squeeze()`
     `k4_y = v + dt*k3_v`
     `y_next = y + (dt/6.0)*(k1_y + 2*k2_y + 2*k3_y + k4_y)`
     `v_next = v + (dt/6.0)*(k1_v + 2*k2_v + 2*k3_v + k4_v)`
     `return y_next, v_next`

5. Fast Truncated Rollout Training (600 steps):
   - Optimizer: Adam(model.parameters(), lr=1e-3, weight_decay=1e-4).
   - 600 iterations, batch_size=32 random starting indices in [0, 3072 - 16], rollout L=16 steps.
   - MSE loss on y_pred vs y_target. Apply `torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)`.

6. Free-Run Evaluation (`with torch.no_grad():`):
   - Pre-allocate `y_sim = torch.zeros(4096, device=device)`, `v_sim = torch.zeros(4096, device=device)`.
   - Set `y_sim[0] = y[0]`, `v_sim[0] = v[0]`.
   - Loop `i` from 0 to 4094: compute RK4 step with scalar `u[i]`.
   - Compute Train RMSE (first 3072) and Test RMSE (last 1024).
   - Print EXACTLY:
     TRAIN RMSE: <value>
     REAL TEST RMSE: <value>

Output ONLY executable Python code inside ```python ... ```."""

    response = llm.invoke([
        SystemMessage(content=sys_prompt),
        HumanMessage(content=f"Instruction: {instruction}\n\nDialogue History:\n{history_str}")
    ])

    raw_code = response.content
    if "```python" in raw_code:
        generated_code = raw_code.split("```python")[1].split("```")[0].strip()
    elif "```" in raw_code:
        generated_code = raw_code.split("```")[1].split("```")[0].strip()
    else:
        generated_code = raw_code.strip()

    exec_result = execute_python_code(generated_code, filename="pinn_silverbox.py")

    return {
        "dialogue": [
            {"role": "Executor (Dynamically Generated Code)", "content": generated_code},
            {"role": "Executor (Execution Result)", "content": exec_result}
        ]
    }


# 5. Router Տրամաբանություն
def router(state: AgentState):
    role = state.get("current_role", "finish")
    if role in ["scientist", "general", "executor", "human"]:
        return role
    else:
        return END


# 6. LangGraph Graph Builder with Checkpointer
builder = StateGraph(AgentState)

builder.add_node("orchestrator", orchestrator_node)
builder.add_node("scientist", scientist_node)
builder.add_node("general", general_node)
builder.add_node("executor", executor_node)
builder.add_node("human", human_node)

builder.set_entry_point("orchestrator")

builder.add_conditional_edges(
    "orchestrator",
    router,
    {
        "scientist": "scientist",
        "general": "general",
        "executor": "executor",
        "human": "human",
        END: END
    }
)

builder.add_edge("scientist", "orchestrator")
builder.add_edge("general", "orchestrator")
builder.add_edge("executor", "orchestrator")
builder.add_edge("human", "orchestrator")

memory = MemorySaver()
app = builder.compile(checkpointer=memory)

print("--- Dynamic Bounded-Output PINN Pipeline Ready ---")

--- Dynamic Bounded-Output PINN Pipeline Ready ---


In [23]:
from langgraph.types import Command

def run_interactive_pipeline(app, initial_state, config):
    print("=== STARTING INTERACTIVE PIPELINE ===")

    # Առաջին execution stream
    for event in app.stream(initial_state, config=config):
        for node, update in event.items():
            print(f"\n--- Node: {node} ---")
            if "dialogue" in update:
                print(f"[{update['dialogue'][-1]['role']}]: {update['dialogue'][-1]['content']}")

    # Պտտվում ենք այնքան անգամ, քանի անգամ Orchestrator-ը interrupt կանի
    while True:
        state = app.get_state(config)

        if not state.next:
            print("\n==========================================")
            print("=== PIPELINE FINISHED SUCCESSFULLY ===")
            print("==========================================")
            break

        print("\n" + "="*50)
        user_reply = input("👉 Enter your reply for Human Node: ")
        print("="*50 + "\n")

        for event in app.stream(Command(resume=user_reply), config=config):
            for node, update in event.items():
                print(f"\n--- Node: {node} ---")
                if "dialogue" in update:
                    print(f"[{update['dialogue'][-1]['role']}]: {update['dialogue'][-1]['content']}")

In [27]:
config_a = {"configurable": {"thread_id": "run_a_silverbox_physics"}}

initial_state_a = {
    "dialogue": [
        {
            "role": "User",
            "content": "Evaluate model performance for the Silverbox dataset."
        }
    ],
    "current_role": "",
    "instruction": "",
    "reason": "",
    "question_to_ask": "",
    "step_count": 0
}

# "Evaluate free-run train and test RMSE on SNLS80mV.mat using V1 for u and V2 for y."
run_interactive_pipeline(app, initial_state_a, config_a)

=== STARTING INTERACTIVE PIPELINE ===

--- Node: orchestrator ---
[Orchestrator]: Decision -> Role: human | Reason: The request to evaluate model performance is ambiguous without specified metrics.

[GRAPH INTERRUPTED - QUESTION FOR HUMAN]: Could you please specify which metrics you would like to use for evaluating the model performance on the Silverbox dataset?

--- Node: __interrupt__ ---

👉 Enter your reply for Human Node: Evaluate free-run train and test RMSE on SNLS80mV.mat using V1 for u and V2 for y


[GRAPH INTERRUPTED - QUESTION FOR HUMAN]: Could you please specify which metrics you would like to use for evaluating the model performance on the Silverbox dataset?

--- Node: human ---
[human]: Evaluate free-run train and test RMSE on SNLS80mV.mat using V1 for u and V2 for y

--- Node: orchestrator ---
[Orchestrator]: Decision -> Role: executor | Reason: The human has provided specific metrics and dataset details for evaluation, allowing the executor to proceed with the task.

--